# AI Travel Planner & Recommender System
**Course:** Introduction to AI — Semester 2, 2025-2026  
**University:** Ho Chi Minh City University of Technology, VNU-HCM  
**Instructor:** Dr. Truong Vinh Lan  

---

## This Notebook Includes:
1. Install Libraries & Download Data
2. Load & Clean Data
3. EDA (Exploratory Data Analysis)
4. Feature Engineering
5. Export Features (.npy / .csv)

**Note:** Go to Runtime → Run All to execute the entire notebook.

## 0. Setup — Install Libraries & Clone Repo

In [ ]:
# Cai dat thu vien can thiet
!pip install opendatasets pandas numpy matplotlib seaborn scikit-learn -q

In [ ]:
# Clone repository (neu chay tren Colab)
import os
if not os.path.exists('Introduction-AI-Assignment'):
    !git clone https://github.com/<YOUR_USERNAME>/Introduction-AI-Assignment.git
    os.chdir('Introduction-AI-Assignment')
else:
    os.chdir('Introduction-AI-Assignment')

print('Working directory:', os.getcwd())

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='husl')
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)

print('Libraries loaded.')

In [ ]:
# Import module data_pipeline
import sys
sys.path.insert(0, '.')

from modules.data_pipeline import (
    download_all_datasets, ensure_dirs,
    load_vietnam_weather, load_hotel_reviews, load_travel_ratings,
    load_traveler_trips, load_world_cities,
    clean_vietnam_weather, clean_hotel_reviews, clean_travel_ratings,
    clean_traveler_trips, clean_world_cities,
    build_distance_matrix, build_cost_matrix, build_travel_time_matrix,
    build_weather_probability_table, build_places_dataframe,
    save_cleaned, save_features, save_feature_csv,
    VN_TOURIST_PLACES, VN_PROVINCE_COORDS,
    RAW_DIR, CLEANED_DIR, PROCESSED_DIR, FEATURES_DIR,
    haversine
)

ensure_dirs()
print('Module data_pipeline imported.')

---
## 1. Download Datasets from Kaggle

Datasets used:
| # | Dataset | Size | AI Component |
|---|---------|------|--------------|
| 1 | Vietnam Weather Data | 181,960 rows | (C) + (D) |
| 2 | 515K Hotel Reviews Europe | 515,000 rows | (E) |
| 3 | Travel Review Ratings (UCI) | 5,456 rows | (E) |
| 4 | Traveler Trip Data | ~21,000 rows | (B) |
| 5 | Worldwide Travel Cities | 560 rows | (E) |

In [ ]:
# Download tat ca datasets tu Kaggle
# Khi chay lan dau, can nhap Kaggle username va API key
# Lay tai: https://www.kaggle.com/settings -> API -> Create New Token
download_all_datasets(use_opendatasets=True)

In [ ]:
# Kiem tra file da tai
for root, dirs, files in os.walk(RAW_DIR):
    for f in files:
        fpath = os.path.join(root, f)
        size_mb = os.path.getsize(fpath) / (1024*1024)
        print(f'  {os.path.relpath(fpath, RAW_DIR):60s} {size_mb:8.2f} MB')

---
## 2. Load & Clean Data

### 2.1 Vietnam Weather Data (181K records, 40 provinces, 2009-2021)
**Serves:** (C) IF-THEN Rule System + (D) Bayesian Network

In [ ]:
# Load & Clean
df_weather_raw = load_vietnam_weather()
print('\nRaw shape:', df_weather_raw.shape)
print('\nCot:', list(df_weather_raw.columns))
df_weather_raw.head()

In [ ]:
df_weather = clean_vietnam_weather(df_weather_raw)
save_cleaned(df_weather, 'vietnam_weather')

print('\nCleaned shape:', df_weather.shape)
print('\nMissing values:')
print(df_weather.isnull().sum())
print('\nData types:')
print(df_weather.dtypes)
df_weather.head()

In [ ]:
df_weather.describe()

### 2.2 Hotel Reviews (515K reviews)
**Serves:** (E) Machine Learning — sentiment classification

In [ ]:
df_reviews_raw = load_hotel_reviews()
print('\nRaw shape:', df_reviews_raw.shape)
print('\nCot:', list(df_reviews_raw.columns))
df_reviews_raw.head()

In [ ]:
df_reviews = clean_hotel_reviews(df_reviews_raw)
save_cleaned(df_reviews, 'hotel_reviews')

print('\nCleaned shape:', df_reviews.shape)
print('\nMissing values (top):')
print(df_reviews.isnull().sum().sort_values(ascending=False).head(10))
df_reviews.head()

### 2.3 Travel Review Ratings — 24 Categories (5,456 users)
**Serves:** (E) Machine Learning — user classification

In [ ]:
df_ratings_raw = load_travel_ratings()
print('\nRaw shape:', df_ratings_raw.shape)
print('\nCot:', list(df_ratings_raw.columns))
df_ratings_raw.head()

In [ ]:
df_ratings = clean_travel_ratings(df_ratings_raw)
save_cleaned(df_ratings, 'travel_ratings')

print('\nCleaned shape:', df_ratings.shape)
df_ratings.describe()

### 2.4 Traveler Trip Data (~21K trips)
**Serves:** (B) CSP — budget and time constraints

In [ ]:
df_trips_raw = load_traveler_trips()
print('\nRaw shape:', df_trips_raw.shape)
print('\nCot:', list(df_trips_raw.columns))
df_trips_raw.head()

In [ ]:
df_trips = clean_traveler_trips(df_trips_raw)
save_cleaned(df_trips, 'traveler_trips')

print('\nCleaned shape:', df_trips.shape)
df_trips.describe()

### 2.5 Worldwide Travel Cities (560 cities)
**Serves:** (E) Machine Learning — city style classification

In [ ]:
df_cities_raw = load_world_cities()
print('\nRaw shape:', df_cities_raw.shape)
print('\nCot:', list(df_cities_raw.columns))
df_cities_raw.head()

In [ ]:
df_cities = clean_world_cities(df_cities_raw)
save_cleaned(df_cities, 'world_cities')

print('\nCleaned shape:', df_cities.shape)
df_cities.describe()

---
## 3. EDA — Exploratory Data Analysis

### 3.1 Vietnam Weather — Weather Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 3.1a: Phan bo luong mua theo thang
if 'month' in df_weather.columns and 'rain_mm' in df_weather.columns:
    monthly_rain = df_weather.groupby('month')['rain_mm'].mean()
    axes[0, 0].bar(monthly_rain.index, monthly_rain.values, color='steelblue')
    axes[0, 0].set_xlabel('Thang')
    axes[0, 0].set_ylabel('Luong mua trung binh (mm)')
    axes[0, 0].set_title('Luong mua trung binh theo thang (Viet Nam)')
    axes[0, 0].set_xticks(range(1, 13))

# 3.1b: Nhiet do trung binh theo thang
if 'month' in df_weather.columns and 'temp_max' in df_weather.columns:
    monthly_temp = df_weather.groupby('month')[['temp_max', 'temp_min']].mean()
    axes[0, 1].plot(monthly_temp.index, monthly_temp['temp_max'], 'r-o', label='Max')
    axes[0, 1].plot(monthly_temp.index, monthly_temp['temp_min'], 'b-o', label='Min')
    axes[0, 1].set_xlabel('Thang')
    axes[0, 1].set_ylabel('Nhiet do (°C)')
    axes[0, 1].set_title('Nhiet do trung binh theo thang')
    axes[0, 1].legend()
    axes[0, 1].set_xticks(range(1, 13))

# 3.1c: Ti le ngay mua theo tinh (top 15)
if 'province' in df_weather.columns and 'is_rainy' in df_weather.columns:
    rain_by_prov = df_weather.groupby('province')['is_rainy'].mean().sort_values(ascending=False).head(15)
    rain_by_prov.plot(kind='barh', ax=axes[1, 0], color='coral')
    axes[1, 0].set_xlabel('Ti le ngay mua')
    axes[1, 0].set_title('Top 15 tinh co ti le mua cao nhat')

# 3.1d: Phan bo do am
if 'humidity' in df_weather.columns:
    axes[1, 1].hist(df_weather['humidity'].dropna(), bins=50, color='teal', edgecolor='white')
    axes[1, 1].set_xlabel('Do am (%)')
    axes[1, 1].set_ylabel('Tan suat')
    axes[1, 1].set_title('Phan bo do am Viet Nam')

plt.tight_layout()
plt.savefig('data/processed/eda_weather.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: data/processed/eda_weather.png')

In [ ]:
# Correlation matrix cua cac bien thoi tiet
weather_num_cols = ['temp_max', 'temp_min', 'wind_speed', 'rain_mm', 'humidity', 'cloud_cover', 'pressure']
weather_num_cols = [c for c in weather_num_cols if c in df_weather.columns]

if weather_num_cols:
    fig, ax = plt.subplots(figsize=(10, 8))
    corr = df_weather[weather_num_cols].corr()
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax)
    ax.set_title('Ma tran tuong quan — Bien thoi tiet Viet Nam')
    plt.tight_layout()
    plt.savefig('data/processed/eda_weather_corr.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Ti le outdoor_suitable theo mua
if 'season' in df_weather.columns and 'outdoor_suitable' in df_weather.columns:
    season_outdoor = df_weather.groupby('season')['outdoor_suitable'].mean()
    fig, ax = plt.subplots(figsize=(8, 5))
    season_order = ['spring', 'summer', 'autumn', 'winter']
    season_outdoor = season_outdoor.reindex(season_order)
    season_outdoor.plot(kind='bar', color=['green', 'orange', 'brown', 'blue'], ax=ax)
    ax.set_ylabel('Ti le ngay thuan loi cho outdoor')
    ax.set_title('Ti le ngay phu hop hoat dong ngoai troi theo mua')
    ax.set_xticklabels(season_order, rotation=0)
    for i, v in enumerate(season_outdoor.values):
        ax.text(i, v + 0.01, f'{v:.1%}', ha='center', fontweight='bold')
    plt.tight_layout()
    plt.savefig('data/processed/eda_outdoor_season.png', dpi=150, bbox_inches='tight')
    plt.show()

### 3.2 Hotel Reviews — Review Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 3.2a: Phan bo diem danh gia
if 'Reviewer_Score' in df_reviews.columns:
    axes[0, 0].hist(df_reviews['Reviewer_Score'].dropna(), bins=20, color='steelblue', edgecolor='white')
    axes[0, 0].set_xlabel('Diem danh gia')
    axes[0, 0].set_ylabel('So luong')
    axes[0, 0].set_title('Phan bo Reviewer Score (515K reviews)')

# 3.2b: Sentiment distribution
if 'sentiment' in df_reviews.columns:
    sent_counts = df_reviews['sentiment'].value_counts()
    sent_counts.plot(kind='bar', color=['red', 'orange', 'green', 'darkgreen'], ax=axes[0, 1])
    axes[0, 1].set_title('Phan bo Sentiment')
    axes[0, 1].set_ylabel('So luong')
    axes[0, 1].set_xticklabels(sent_counts.index, rotation=0)

# 3.2c: Phan bo so tu trong review
if 'review_word_count' in df_reviews.columns:
    axes[1, 0].hist(df_reviews['review_word_count'].clip(upper=200), bins=50, color='teal', edgecolor='white')
    axes[1, 0].set_xlabel('So tu trong review')
    axes[1, 0].set_ylabel('Tan suat')
    axes[1, 0].set_title('Phan bo so tu (clipped at 200)')

# 3.2d: Top 10 quoc tich reviewer
if 'Reviewer_Nationality' in df_reviews.columns:
    top_nations = df_reviews['Reviewer_Nationality'].str.strip().value_counts().head(10)
    top_nations.plot(kind='barh', color='coral', ax=axes[1, 1])
    axes[1, 1].set_xlabel('So luong reviews')
    axes[1, 1].set_title('Top 10 quoc tich Reviewer')

plt.tight_layout()
plt.savefig('data/processed/eda_reviews.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: data/processed/eda_reviews.png')

### 3.3 Travel Ratings — User Preference Analysis

In [ ]:
# Rating trung binh theo loai hinh
category_cols = [c for c in df_ratings.columns if c not in [
    'User', 'User Id', 'Unnamed: 0', 'top_category', 'avg_rating', 'rating_std', 'num_rated'
]]
# Chi lay cot co kieu so
category_cols = [c for c in category_cols if df_ratings[c].dtype in ['float64', 'int64']]

if category_cols:
    avg_by_cat = df_ratings[category_cols].mean().sort_values(ascending=True)
    fig, ax = plt.subplots(figsize=(10, 8))
    avg_by_cat.plot(kind='barh', color='steelblue', ax=ax)
    ax.set_xlabel('Rating trung binh')
    ax.set_title(f'Rating trung binh theo {len(category_cols)} loai hinh du lich')
    plt.tight_layout()
    plt.savefig('data/processed/eda_ratings.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Correlation heatmap cua cac loai hinh
if len(category_cols) >= 5:
    fig, ax = plt.subplots(figsize=(14, 12))
    corr = df_ratings[category_cols].corr()
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
                ax=ax, xticklabels=True, yticklabels=True)
    ax.set_title('Ma tran tuong quan — Rating cac loai hinh du lich')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig('data/processed/eda_ratings_corr.png', dpi=150, bbox_inches='tight')
    plt.show()

### 3.4 Traveler Trips — Travel Cost Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 3.4a: Phan bo tong chi phi
if 'total_cost' in df_trips.columns:
    axes[0, 0].hist(df_trips['total_cost'].clip(upper=10000), bins=50, color='steelblue', edgecolor='white')
    axes[0, 0].set_xlabel('Tong chi phi ($)')
    axes[0, 0].set_ylabel('So luong')
    axes[0, 0].set_title('Phan bo tong chi phi (clipped at $10,000)')

# 3.4b: Budget level distribution
if 'budget_level' in df_trips.columns:
    budget_counts = df_trips['budget_level'].value_counts()
    budget_counts.plot(kind='bar', color=['green', 'blue', 'orange', 'red'], ax=axes[0, 1])
    axes[0, 1].set_title('Phan bo Budget Level')
    axes[0, 1].set_ylabel('So luong')
    axes[0, 1].set_xticklabels(budget_counts.index, rotation=0)

# 3.4c: Duration distribution
if 'Duration (days)' in df_trips.columns:
    axes[1, 0].hist(df_trips['Duration (days)'].dropna(), bins=30, color='teal', edgecolor='white')
    axes[1, 0].set_xlabel('So ngay')
    axes[1, 0].set_ylabel('Tan suat')
    axes[1, 0].set_title('Phan bo thoi gian du lich')

# 3.4d: Top destinations
if 'Destination' in df_trips.columns:
    top_dest = df_trips['Destination'].value_counts().head(15)
    top_dest.plot(kind='barh', color='coral', ax=axes[1, 1])
    axes[1, 1].set_xlabel('So luong trips')
    axes[1, 1].set_title('Top 15 diem den pho bien')

plt.tight_layout()
plt.savefig('data/processed/eda_trips.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: data/processed/eda_trips.png')

---
## 4. Feature Engineering

### 4.1 Distance Matrix — Vietnam Tourist Spots
Used for (A) A* and (B) CSP

In [ ]:
# Build & save distance matrix
place_names, dist_matrix = build_distance_matrix()
save_features(dist_matrix, 'distance_matrix')

# Hien thi
df_dist = pd.DataFrame(dist_matrix, index=place_names, columns=place_names)
print(f'Ma tran khoang cach: {dist_matrix.shape[0]} x {dist_matrix.shape[1]} diem')
print(f'Khoang cach min (> 0): {dist_matrix[dist_matrix > 0].min():.1f} km')
print(f'Khoang cach max: {dist_matrix.max():.1f} km')
df_dist.head(10)

In [ ]:
# Visualize distance matrix (heatmap)
fig, ax = plt.subplots(figsize=(18, 16))
sns.heatmap(df_dist, cmap='YlOrRd', ax=ax, fmt='.0f',
            xticklabels=True, yticklabels=True)
ax.set_title(f'Ma tran khoang cach (km) — {len(place_names)} diem du lich VN')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.savefig('data/processed/distance_matrix_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cost matrix & Travel time matrix
_, cost_matrix = build_cost_matrix()
save_features(cost_matrix, 'cost_matrix')

_, time_matrix = build_travel_time_matrix()
save_features(time_matrix, 'travel_time_matrix')

print(f'Cost matrix range: {cost_matrix[cost_matrix > 0].min():,.0f} — {cost_matrix.max():,.0f} VND')
print(f'Time matrix range: {time_matrix[time_matrix > 0].min():.1f} — {time_matrix.max():.1f} hours')

### 4.2 Places DataFrame — Tourist Spot Information

In [ ]:
df_places = build_places_dataframe()
save_feature_csv(df_places, 'vn_tourist_places')

print(f'\nSo diem du lich: {len(df_places)}')
print(f'\nPhan bo theo category:')
print(df_places['category'].value_counts())
print(f'\nPhan bo theo tinh:')
print(df_places['province'].value_counts())
df_places

In [ ]:
# Map cac diem du lich (scatter plot)
fig, ax = plt.subplots(figsize=(10, 14))
colors = {'culture': 'red', 'nature': 'green', 'beach': 'blue',
          'entertainment': 'purple', 'adventure': 'orange'}
for cat, color in colors.items():
    subset = df_places[df_places['category'] == cat]
    ax.scatter(subset['longitude'], subset['latitude'],
               c=color, label=cat, s=80, edgecolors='white', zorder=5)
    for _, row in subset.iterrows():
        ax.annotate(row['place_name'], (row['longitude'], row['latitude']),
                     fontsize=6, ha='left', va='bottom')

ax.set_xlabel('Kinh do (Longitude)')
ax.set_ylabel('Vi do (Latitude)')
ax.set_title(f'Ban do {len(df_places)} diem du lich Viet Nam')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('data/processed/places_map.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.3 Weather Probability Table — For Bayesian Network

In [ ]:
weather_probs = build_weather_probability_table(df_weather)
save_feature_csv(weather_probs, 'weather_probabilities')

print(f'\nBang xac suat: {weather_probs.shape[0]} dong (tinh x thang)')
print(f'\nVi du — Ha Noi:')
weather_probs[weather_probs['province'].str.contains('Ha Noi|Hanoi|Ha noi', case=False, na=False)]

In [ ]:
# Heatmap xac suat mua theo tinh x thang (top 15 tinh du lich)
tourist_provinces = ['Ha Noi', 'Ho Chi Minh', 'Da Nang', 'Hue',
                     'Khanh Hoa', 'Lam Dong', 'Quang Ninh', 'Quang Binh',
                     'Quang Nam', 'Binh Thuan', 'Kien Giang', 'Lao Cai',
                     'Ninh Binh', 'Hai Phong', 'Can Tho']

# Tim cac tinh co trong du lieu
available_provinces = []
for tp in tourist_provinces:
    matches = weather_probs[weather_probs['province'].str.contains(tp, case=False, na=False)]
    if len(matches) > 0:
        available_provinces.append(matches['province'].iloc[0])

if available_provinces:
    subset = weather_probs[weather_probs['province'].isin(available_provinces)]
    pivot = subset.pivot_table(index='province', columns='month', values='p_rain')
    
    fig, ax = plt.subplots(figsize=(14, 8))
    sns.heatmap(pivot, annot=True, fmt='.2f', cmap='Blues', ax=ax)
    ax.set_title('Xac suat mua P(rain | province, month) — Cac tinh du lich')
    ax.set_xlabel('Thang')
    ax.set_ylabel('Tinh')
    plt.tight_layout()
    plt.savefig('data/processed/weather_rain_prob_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Khong tim thay tinh du lich trong du lieu. Kiem tra ten tinh.')

### 4.4 Text Features — From Hotel Reviews for ML

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Lay sample 50K reviews de TF-IDF (tiet kiem RAM)
if 'full_review' in df_reviews.columns:
    sample_reviews = df_reviews[['full_review', 'sentiment_binary', 'Reviewer_Score']].dropna()
    sample_reviews = sample_reviews[sample_reviews['full_review'].str.len() > 20]
    
    if len(sample_reviews) > 50000:
        sample_reviews = sample_reviews.sample(50000, random_state=42)
    
    print(f'Sample size: {len(sample_reviews):,} reviews')
    
    # TF-IDF vectorization (top 5000 features)
    tfidf = TfidfVectorizer(max_features=5000, stop_words='english', ngram_range=(1, 2))
    X_tfidf = tfidf.fit_transform(sample_reviews['full_review'])
    
    print(f'TF-IDF matrix: {X_tfidf.shape}')
    print(f'Top 20 features: {tfidf.get_feature_names_out()[:20].tolist()}')
    
    # Save
    from scipy import sparse
    sparse.save_npz('data/features/review_tfidf.npz', X_tfidf)
    np.save('data/features/review_labels.npy', sample_reviews['sentiment_binary'].values)
    np.save('data/features/review_scores.npy', sample_reviews['Reviewer_Score'].values)
    
    # Save TF-IDF feature names
    pd.DataFrame({'feature': tfidf.get_feature_names_out()}).to_csv(
        'data/features/tfidf_features.csv', index=False)
    
    print(f'\nSaved:')
    print(f'  data/features/review_tfidf.npz ({X_tfidf.shape})')
    print(f'  data/features/review_labels.npy ({len(sample_reviews):,})')
    print(f'  data/features/review_scores.npy ({len(sample_reviews):,})')
else:
    print('Cot full_review khong ton tai. Kiem tra lai.')

### 4.5 User Feature Vectors — From Travel Ratings for User Classification

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Lay cac cot rating
cat_cols = [c for c in df_ratings.columns if c not in [
    'User', 'User Id', 'Unnamed: 0', 'top_category', 'avg_rating', 'rating_std', 'num_rated'
] and df_ratings[c].dtype in ['float64', 'int64']]

if cat_cols:
    X_ratings = df_ratings[cat_cols].values
    
    # Standardize
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_ratings)
    
    # K-Means clustering de tao nhan traveler_type
    # Chon k=5 (5 loai khach du lich)
    kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    
    df_ratings['cluster'] = labels
    
    # Dat ten cluster dua tren dac trung trung binh
    cluster_means = df_ratings.groupby('cluster')[cat_cols].mean()
    print('Dac trung trung binh cua cac cluster:')
    print(cluster_means.T)
    
    # Save
    np.save('data/features/user_features.npy', X_scaled)
    np.save('data/features/user_cluster_labels.npy', labels)
    save_cleaned(df_ratings, 'travel_ratings_clustered')
    
    print(f'\nPhan bo clusters:')
    print(df_ratings['cluster'].value_counts().sort_index())
    print(f'\nSaved: user_features.npy ({X_scaled.shape}), user_cluster_labels.npy ({len(labels)})')

In [ ]:
# Visualize clusters (PCA 2D)
from sklearn.decomposition import PCA

if 'cluster' in df_ratings.columns:
    pca = PCA(n_components=2, random_state=42)
    X_2d = pca.fit_transform(X_scaled)
    
    fig, ax = plt.subplots(figsize=(10, 8))
    scatter = ax.scatter(X_2d[:, 0], X_2d[:, 1], c=labels, cmap='tab10',
                          alpha=0.6, s=20, edgecolors='white', linewidth=0.3)
    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
    ax.set_title('Traveler Clusters (PCA 2D projection)')
    plt.colorbar(scatter, label='Cluster')
    plt.tight_layout()
    plt.savefig('data/processed/user_clusters_pca.png', dpi=150, bbox_inches='tight')
    plt.show()

---
## 5. Summary of Processed Data

In [ ]:
print('=' * 70)
print('TONG KET DU LIEU DA XU LY')
print('=' * 70)

# Kiem tra cleaned files
print('\n--- Cleaned Data ---')
for root, dirs, files in os.walk(CLEANED_DIR):
    for f in sorted(files):
        fpath = os.path.join(root, f)
        size_mb = os.path.getsize(fpath) / (1024*1024)
        if f.endswith('.csv'):
            nrows = sum(1 for _ in open(fpath)) - 1
            print(f'  {f:40s} {size_mb:8.2f} MB   {nrows:>10,} rows')
        else:
            print(f'  {f:40s} {size_mb:8.2f} MB')

# Kiem tra feature files
print('\n--- Feature Files ---')
for root, dirs, files in os.walk(FEATURES_DIR):
    for f in sorted(files):
        fpath = os.path.join(root, f)
        size_mb = os.path.getsize(fpath) / (1024*1024)
        if f.endswith('.npy'):
            arr = np.load(fpath)
            print(f'  {f:40s} {size_mb:8.2f} MB   shape: {arr.shape}')
        elif f.endswith('.npz'):
            print(f'  {f:40s} {size_mb:8.2f} MB   (sparse matrix)')
        else:
            print(f'  {f:40s} {size_mb:8.2f} MB')

# Kiem tra processed images
print('\n--- EDA Plots ---')
for root, dirs, files in os.walk(PROCESSED_DIR):
    for f in sorted(files):
        if f.endswith('.png'):
            fpath = os.path.join(root, f)
            size_kb = os.path.getsize(fpath) / 1024
            print(f'  {f:40s} {size_kb:8.1f} KB')

print('\n' + '=' * 70)
print('HOAN TAT - Du lieu san sang cho cac thanh phan AI (A)-(E)')
print('=' * 70)

---
## Data Mapping → AI Components

| Feature File | AI Component | Description |
|---|---|---|
| `distance_matrix.npy` | **(A) A* Search** | Distance matrix for 30 Vietnam tourist spots |
| `cost_matrix.npy` | **(A) Search** + **(B) CSP** | Travel cost matrix (VND) |
| `travel_time_matrix.npy` | **(A) Search** + **(B) CSP** | Travel time matrix (hours) |
| `vn_tourist_places.csv` | **(A)(B)** | Place info (opening hours, entry fee, category) |
| `vietnam_weather.csv` | **(C) IF-THEN** + **(D) Bayes** | Weather data — 181K rows |
| `weather_probabilities.csv` | **(D) Bayesian Network** | P(rain\|province,month), P(outdoor\|...) |
| `traveler_trips.csv` | **(B) CSP** | Budget, time, and activity type constraints |
| `review_tfidf.npz` | **(E) ML** | TF-IDF vectors for 50K reviews |
| `review_labels.npy` | **(E) ML** | Sentiment labels (binary) |
| `user_features.npy` | **(E) ML** | Feature vectors — 5,456 users × 24 categories |
| `user_cluster_labels.npy` | **(E) ML** | Cluster labels (5 traveler groups) |